# 🖼️ Image Captioning — Model A: YOLO + BLIP + CLIP (FAISS Retrieval)
### COCO val2017 evaluation with BLEU, CIDEr, visual samples & comparison-ready results

In [ ]:
!pip install -q ultralytics transformers faiss-cpu pycocoevalcap pillow tqdm nltk matplotlib seaborn

In [ ]:
!wget -q http://images.cocodataset.org/zips/val2017.zip
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q val2017.zip
!unzip -q annotations_trainval2017.zip

In [ ]:
import os, json, time, torch, numpy as np
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns

from ultralytics import YOLO
from transformers import BlipProcessor, BlipForConditionalGeneration
from transformers import CLIPProcessor, CLIPModel
import faiss
from sklearn.metrics.pairwise import cosine_similarity

# Styling
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#e0e0e0',
    'text.color':       '#e0e0e0',
    'xtick.color':      '#aaa',
    'ytick.color':      '#aaa',
    'grid.color':       '#333',
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
})
ACCENT = '#7c83fd'
ACCENT2 = '#f7971e'
print('✅ Imports ready')

In [ ]:
## ── Section 1: Load Models ──
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

processor  = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
blip_model = BlipForConditionalGeneration.from_pretrained(
    'Salesforce/blip-image-captioning-base').to(device)
blip_model.eval()

clip_model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()

yolo = YOLO('yolov8n.pt')
print('✅ All models loaded')

In [ ]:
## ── Section 2: Load COCO Annotations ──
ANN_FILE = 'annotations/captions_val2017.json'
with open(ANN_FILE) as f:
    data = json.load(f)

image_id_to_name = {img['id']: img['file_name'] for img in data['images']}
caption_db, image_to_captions = [], {}
for ann in data['annotations']:
    name = image_id_to_name[ann['image_id']]
    cap  = ann['caption']
    caption_db.append(cap)
    image_to_captions.setdefault(name, []).append(cap)
print(f'✅ {len(image_to_captions)} images, {len(caption_db)} captions loaded')

In [ ]:
## ── Section 3: CLIP Embeddings & FAISS Index ──
def get_clip_text_emb(texts, batch_size=32):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = clip_processor(text=batch, return_tensors='pt',
                                padding=True, truncation=True).to(device)
        with torch.no_grad():
            emb = clip_model.get_text_features(**inputs)
        all_embs.append(emb.detach().cpu().numpy())
        del inputs, emb
        torch.cuda.empty_cache()
    return np.vstack(all_embs)

def get_clip_img(img):
    inputs = clip_processor(images=img, return_tensors='pt').to(device)
    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs)
    return emb.detach().cpu().numpy()

caption_db_sub = caption_db[:50000]
text_embs = get_clip_text_emb(caption_db_sub, batch_size=32)
faiss.normalize_L2(text_embs)
index = faiss.IndexFlatIP(text_embs.shape[1])
index.add(text_embs)
print(f'✅ FAISS index built — {index.ntotal} vectors')

In [ ]:
## ── Section 4: Captioning Pipeline ──
def yolo_objects(img_path, conf=0.5):
    res   = yolo(img_path, verbose=False)[0]
    names = yolo.names
    objs  = []
    if res.boxes is not None:
        for c, cf in zip(res.boxes.cls, res.boxes.conf):
            if cf > conf:
                objs.append(names[int(c)])
    return list(set(objs))[:5], res

def generate_blip(img):
    inputs = processor(images=img, return_tensors='pt').to(device)
    out    = blip_model.generate(**inputs, max_new_tokens=30)
    return processor.decode(out[0], skip_special_tokens=True)

def retrieve(img, k=5):
    emb  = get_clip_img(img)
    D, I = index.search(emb, k)
    return [caption_db_sub[i] for i in I[0]]

def caption_image(img_path):
    img        = Image.open(img_path).convert('RGB')
    blip_cap   = generate_blip(img)
    objs, yres = yolo_objects(img_path)
    yolo_cap   = 'a photo of ' + ', '.join(objs) if objs else ''
    retrieved  = retrieve(img)
    candidates = [blip_cap, yolo_cap] + retrieved
    img_emb    = get_clip_img(img)
    txt_emb    = get_clip_text_emb(candidates, batch_size=8)
    sims       = cosine_similarity(img_emb, txt_emb)[0]
    best       = candidates[np.argmax(sims)]
    return best, objs, yres, blip_cap

def caption_with_latency(img_path):
    t0 = time.time()
    result = caption_image(img_path)
    return result, time.time() - t0

print('✅ Pipeline functions defined')

In [ ]:
## ── Section 5: Visual Sample Predictions ──
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

N_SAMPLES = 6
sample_imgs = list(image_to_captions.keys())[:N_SAMPLES]
VAL_IMG = 'val2017'

fig = plt.figure(figsize=(20, 14), facecolor='#0d0d1a')
fig.suptitle('Model A — YOLO + BLIP + CLIP  |  Sample Predictions', 
             fontsize=18, color='white', fontweight='bold', y=1.01)

for idx, img_name in enumerate(sample_imgs):
    img_path = os.path.join(VAL_IMG, img_name)
    try:
        (best_cap, objs, yres, blip_cap), lat = caption_with_latency(img_path)
        gt_cap = image_to_captions[img_name][0]

        ax = fig.add_subplot(2, 3, idx + 1)
        annotated = yres.plot()
        ax.imshow(annotated[:, :, ::-1])
        ax.axis('off')

        title_text = f'🔮 {best_cap}\n📦 YOLO: {objs}\n⏱ {lat:.2f}s'
        ax.set_title(title_text, fontsize=8, color='#e0e0e0',
                     pad=6, loc='left', wrap=True)
        # Ground-truth annotation below image
        ax.set_xlabel(f'GT: {gt_cap[:70]}...', fontsize=7,
                      color='#aaffaa', labelpad=4)
    except Exception as e:
        print(f'Skip {img_name}: {e}')

plt.tight_layout()
plt.savefig('modelA_samples.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d1a')
plt.show()
print('✅ Visual samples saved to modelA_samples.png')

In [ ]:
## ── Section 6: Full Evaluation (BLEU + CIDEr + Latency) ──
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider

gts, res = {}, {}
latencies = []
imgs = list(image_to_captions.keys())[:100]

print('Running evaluation on 100 images...\n')
for img_name in tqdm(imgs):
    try:
        path = os.path.join(VAL_IMG, img_name)
        (pred, _, _, _), lat = caption_with_latency(path)
        latencies.append(lat)
        refs = [str(r) for r in image_to_captions[img_name] if len(r) > 0]
        if not refs: continue
        gts[img_name] = refs
        res[img_name] = [str(pred)]
    except Exception as e:
        continue

bleu   = Bleu(4)
bscore, _ = bleu.compute_score(gts, res)
cider  = Cider()
cscore, _ = cider.compute_score(gts, res)

# Store for comparison section
modelA_results = {
    'BLEU-1': bscore[0], 'BLEU-2': bscore[1],
    'BLEU-3': bscore[2], 'BLEU-4': bscore[3],
    'CIDEr':  cscore,
    'Avg_Latency': np.mean(latencies),
    'Min_Latency': np.min(latencies),
    'Max_Latency': np.max(latencies),
}

print('\n══════════════════════════════════')
print('  MODEL A — EVALUATION RESULTS')
print('══════════════════════════════════')
for k, v in modelA_results.items():
    print(f'  {k:<18}: {v:.4f}')
print('══════════════════════════════════')

In [ ]:
## ── Section 7: Metric Visualizations ──

fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor='#0d0d1a')
fig.suptitle('Model A — YOLO + BLIP + CLIP  |  Evaluation Metrics',
             fontsize=16, color='white', fontweight='bold')

# ── Plot 1: BLEU bars ──
ax = axes[0]
bleu_names = ['BLEU-1','BLEU-2','BLEU-3','BLEU-4']
bleu_vals  = [modelA_results[k] for k in bleu_names]
colors = ['#7c83fd','#5c6bc0','#3949ab','#1a237e']
bars = ax.bar(bleu_names, bleu_vals, color=colors, edgecolor='#333', width=0.6)
for bar, v in zip(bars, bleu_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{v:.3f}', ha='center', va='bottom', fontsize=10, color='white')
ax.set_ylim(0, max(bleu_vals)*1.3)
ax.set_title('BLEU Scores', color='white', fontsize=13)
ax.set_ylabel('Score', color='white')
ax.grid(axis='y', alpha=0.3)

# ── Plot 2: CIDEr gauge ──
ax2 = axes[1]
cider_val = modelA_results['CIDEr']
wedge_colors = ['#f7971e', '#333']
sizes = [min(cider_val, 1.5), max(0, 1.5 - cider_val)]
wedges, _ = ax2.pie(sizes, colors=wedge_colors, startangle=90,
                    wedgeprops={'width': 0.45, 'edgecolor': '#0d0d1a', 'linewidth': 3})
ax2.text(0, 0, f'{cider_val:.3f}', ha='center', va='center',
         fontsize=22, color='#f7971e', fontweight='bold')
ax2.set_title('CIDEr Score\n(out of 1.5 typical range)', color='white', fontsize=13)

# ── Plot 3: Latency distribution ──
ax3 = axes[2]
ax3.hist(latencies, bins=20, color='#7c83fd', edgecolor='#333', alpha=0.85)
ax3.axvline(np.mean(latencies), color='#f7971e', linestyle='--',
            linewidth=2, label=f'Mean={np.mean(latencies):.2f}s')
ax3.set_title('Inference Latency Distribution', color='white', fontsize=13)
ax3.set_xlabel('Time (s)', color='white')
ax3.set_ylabel('Count', color='white')
ax3.legend(facecolor='#1a1a2e', labelcolor='white')
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('modelA_metrics.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()
print('✅ Metric plots saved to modelA_metrics.png')

In [ ]:
## ── Section 8: Per-Image Score Analysis ──
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

per_bleu, per_keys = [], []
for k in list(gts.keys())[:50]:
    refs_tok = [r.lower().split() for r in gts[k]]
    hyp_tok  = res[k][0].lower().split()
    sc = sentence_bleu(refs_tok, hyp_tok,
                       smoothing_function=SmoothingFunction().method1)
    per_bleu.append(sc)
    per_keys.append(k[:10])

fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor='#0d0d1a')
fig.suptitle('Model A — Per-Image BLEU Analysis (first 50 images)',
             color='white', fontsize=14, fontweight='bold')

# Line chart
axes[0].plot(range(len(per_bleu)), per_bleu, color='#7c83fd', linewidth=1.5, alpha=0.8)
axes[0].fill_between(range(len(per_bleu)), per_bleu, alpha=0.25, color='#7c83fd')
axes[0].axhline(np.mean(per_bleu), color='#f7971e', linestyle='--',
                label=f'Mean={np.mean(per_bleu):.3f}')
axes[0].set_title('Per-Image BLEU-1 Trend', color='white', fontsize=12)
axes[0].set_xlabel('Image Index', color='white')
axes[0].set_ylabel('BLEU-1', color='white')
axes[0].legend(facecolor='#1a1a2e', labelcolor='white')
axes[0].grid(alpha=0.3)

# Score distribution
axes[1].hist(per_bleu, bins=15, color='#f7971e', edgecolor='#333', alpha=0.85)
axes[1].set_title('BLEU-1 Score Distribution', color='white', fontsize=12)
axes[1].set_xlabel('BLEU-1', color='white')
axes[1].set_ylabel('Frequency', color='white')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('modelA_bleu_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()
print(f'✅ Mean per-image BLEU-1: {np.mean(per_bleu):.4f}')

In [ ]:
## ── Save results for comparison ──
import pickle
with open('modelA_results.pkl', 'wb') as f:
    pickle.dump({'metrics': modelA_results, 'per_bleu': per_bleu,
                 'latencies': latencies}, f)
print('✅ Model A results saved to modelA_results.pkl')